# VeAg Model Client - Interactive Deployment Notebook

## Overview

This notebook demonstrates the **VeAg crop disease detection system** deployment and testing process. It creates a Gradio web interface for:
- **Multi-model predictions** (ConvNeXt, EfficientNetV2, DeiT)
- **Ensemble inference** with customizable weights
- **AI-powered treatment advice** via Google Gemini API
- **Comprehensive visualizations** (charts, gauges, pie charts)

### Current Example: Rice Leaf Disease Detection

This example uses **rice leaf disease detection** but the same architecture can be adapted for other crops (wheat, corn, tomato, potato, etc.).

---

## Prerequisites

Before running this notebook:
1. Train models using `model/backend/ML_Crop_Disease_Detection_Model.ipynb`
2. Copy trained `.pth` files to `models/checkpoints/`
3. Update `classes.json` with your disease categories
4. (Optional) Set up Gemini API key in `.env` file

---

## What This Notebook Does

1. **Setup**: Creates project structure and installs dependencies
2. **Model Handlers**: Defines model loading and ensemble logic
3. **Prediction Engine**: Implements inference functions
4. **Gradio Interface**: Launches web UI for predictions
5. **Testing**: Validates all components work correctly

---

## Quick Start

Run all cells in sequence to launch the Gradio application. The web interface will be available at `http://localhost:7860`.

In [4]:
%%writefile requirements.txt
gradio>=4.26.0
torch
torchvision
timm
pandas
pillow
numpy
python-dotenv
matplotlib

Writing requirements.txt


## Step 1: Create Dependencies File

Creating `requirements.txt` with all necessary packages for the VeAg model client.

In [8]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Step 2: Install Dependencies

Installing all required packages. This may take a few minutes on first run.

In [3]:
%%writefile classes.json
["Bacterial leaf blight", "Brown spot", "Leaf smut"]

Overwriting classes.json


## Step 3: Define Disease Classes

Creating `classes.json` with disease categories. 

**Important:** These classes must match the number of classes your trained models expect!

**Current Example:** Rice leaf diseases
- Bacterial leaf blight
- Brown spot
- Leaf smut

**To Adapt for Other Crops:** Replace these with your crop-specific diseases (e.g., for tomato: early blight, late blight, leaf mold, healthy).

In [3]:
directories = [
    'src',
    'models/checkpoints',
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

open('src/__init__.py', 'a').close()

print("Project structure created successfully!")
print("\nDirectory tree:")
os.system('apt-get install tree -y > /dev/null 2>&1')
os.system('tree -L 3 -I "*.ipynb_checkpoints"')

Project structure created successfully!

Directory tree:


0

## Step 4: Create Project Structure

Setting up directories for:
- `src/`: Source code modules (model handlers, prediction functions)
- `models/checkpoints/`: Trained model files (.pth files go here)

**Important:** After training models in `model/backend/`, copy the generated `.pth` files to `models/checkpoints/` directory.

In [7]:
%%writefile src/model_handler.py
"""
Model Handler Module for RiceVision Project
Handles model architectures, ensemble logic, loading, and saving
"""

import torch
import torch.nn as nn
import timm
from pathlib import Path
import json
from datetime import datetime

def get_model(model_name, num_classes, pretrained=True):
    """
    Get a pretrained model by name

    Args:
        model_name: One of ['ConvNeXt-Base', 'EfficientNetV2-M', 'DeiT-Small']
        num_classes: Number of output classes
        pretrained: Whether to load pretrained weights

    Returns:
        PyTorch model
    """
    print(f"Creating model: {model_name}")
    print(f"   Num classes: {num_classes}")
    print(f"   Pretrained: {pretrained}")

    if model_name == 'ConvNeXt-Base':
        model = timm.create_model(
            'convnext_base',
            pretrained=pretrained,
            num_classes=num_classes
        )
        print(f"ConvNeXt-Base loaded (88M parameters)")

    elif model_name == 'EfficientNetV2-M':
        model = timm.create_model(
            'tf_efficientnetv2_m',
            pretrained=pretrained,
            num_classes=num_classes
        )
        print(f"EfficientNetV2-M loaded (54M parameters)")

    elif model_name == 'DeiT-Small':
        model = timm.create_model(
            'deit_small_patch16_224',
            pretrained=pretrained,
            num_classes=num_classes
        )
        print(f"DeiT-Small loaded (22M parameters)")

    else:
        raise ValueError(f"Unsupported model name: {model_name}. "
                        f"Choose from ['ConvNeXt-Base', 'EfficientNetV2-M', 'DeiT-Small']")

    return model


def count_parameters(model):
    """Count trainable parameters in a model"""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        'total': total,
        'trainable': trainable,
        'total_millions': total / 1e6,
        'trainable_millions': trainable / 1e6
    }


def get_model_info(model, model_name):
    """Get comprehensive information about a model"""
    params = count_parameters(model)

    info = {
        'name': model_name,
        'total_parameters': params['total'],
        'trainable_parameters': params['trainable'],
        'total_parameters_M': f"{params['total_millions']:.2f}M",
        'trainable_parameters_M': f"{params['trainable_millions']:.2f}M",
    }

    return info


class EnsembleModel(nn.Module):
    """
    Ensemble of three expert models with weighted averaging
    """
    def __init__(self, model_convnext, model_efficientnet, model_deit,
                 weights=None, mode='average'):
        """
        Args:
            model_convnext: ConvNeXt-Base model
            model_efficientnet: EfficientNetV2-M model
            model_deit: DeiT-Small model
            weights: Optional list of 3 weights for weighted averaging
            mode: 'average' or 'weighted' or 'max'
        """
        super().__init__()

        self.model_convnext = model_convnext
        self.model_efficientnet = model_efficientnet
        self.model_deit = model_deit

        self.mode = mode

        if weights is None:
            self.weights = [1/3, 1/3, 1/3]
        else:
            assert len(weights) == 3, "Must provide exactly 3 weights"
            assert abs(sum(weights) - 1.0) < 1e-6, "Weights must sum to 1.0"
            self.weights = weights

        print(f"🎯 Ensemble created with mode: {mode}")
        print(f"   Weights: ConvNeXt={self.weights[0]:.3f}, "
              f"EfficientNet={self.weights[1]:.3f}, DeiT={self.weights[2]:.3f}")

    def forward(self, x):
        """
        Forward pass through all models and combine predictions
        """
        out_convnext = self.model_convnext(x)
        out_efficientnet = self.model_efficientnet(x)
        out_deit = self.model_deit(x)

        if self.mode == 'average':
            prob_convnext = torch.softmax(out_convnext, dim=1)
            prob_efficientnet = torch.softmax(out_efficientnet, dim=1)
            prob_deit = torch.softmax(out_deit, dim=1)

            out = (prob_convnext + prob_efficientnet + prob_deit) / 3

        elif self.mode == 'weighted':
            prob_convnext = torch.softmax(out_convnext, dim=1)
            prob_efficientnet = torch.softmax(out_efficientnet, dim=1)
            prob_deit = torch.softmax(out_deit, dim=1)

            out = (self.weights[0] * prob_convnext +
                   self.weights[1] * prob_efficientnet +
                   self.weights[2] * prob_deit)

        elif self.mode == 'max':
            prob_convnext = torch.softmax(out_convnext, dim=1)
            prob_efficientnet = torch.softmax(out_efficientnet, dim=1)
            prob_deit = torch.softmax(out_deit, dim=1)

            stacked = torch.stack([prob_convnext, prob_efficientnet, prob_deit])
            out, _ = torch.max(stacked, dim=0)

        else:
            raise ValueError(f"Unsupported ensemble mode: {self.mode}")

        return out

    def get_individual_predictions(self, x):
        """
        Get predictions from each individual model separately
        Returns dict with all predictions
        """
        with torch.no_grad():
            out_convnext = torch.softmax(self.model_convnext(x), dim=1)
            out_efficientnet = torch.softmax(self.model_efficientnet(x), dim=1)
            out_deit = torch.softmax(self.model_deit(x), dim=1)

        return {
            'ConvNeXt-Base': out_convnext,
            'EfficientNetV2-M': out_efficientnet,
            'DeiT-Small': out_deit,
            'Ensemble': self.forward(x)
        }


def create_ensemble(num_classes, pretrained=True, weights=None, mode='average'):
    """
    Create an ensemble of all three expert models

    Args:
        num_classes: Number of output classes
        pretrained: Whether to use pretrained weights
        weights: Optional list of 3 weights for ensemble
        mode: Ensemble combination mode

    Returns:
        EnsembleModel instance
    """
    print("Creating Ensemble Model...")
    print("="*60)

    model_convnext = get_model('ConvNeXt-Base', num_classes, pretrained)
    model_efficientnet = get_model('EfficientNetV2-M', num_classes, pretrained)
    model_deit = get_model('DeiT-Small', num_classes, pretrained)

    ensemble = EnsembleModel(
        model_convnext,
        model_efficientnet,
        model_deit,
        weights=weights,
        mode=mode
    )

    print("="*60)
    print("Ensemble created successfully!")

    total_params = (count_parameters(model_convnext)['total'] +
                   count_parameters(model_efficientnet)['total'] +
                   count_parameters(model_deit)['total'])

    print(f"Total ensemble parameters: {total_params/1e6:.2f}M")

    return ensemble


def save_model(model, model_name, run_name, metadata=None):
    """
    Save model weights and metadata

    Args:
        model: PyTorch model to save
        model_name: Architecture name
        run_name: Unique run identifier
        metadata: Optional dict with additional info

    Returns:
        Path to saved checkpoint
    """
    checkpoint_dir = Path('models/checkpoints')
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = checkpoint_dir / f"{run_name}.pth"

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'model_name': model_name,
        'run_name': run_name,
        'timestamp': datetime.now().isoformat(),
    }

    if metadata:
        checkpoint['metadata'] = metadata

    torch.save(checkpoint, checkpoint_path)

    print(f"Model saved to: {checkpoint_path}")

    return str(checkpoint_path)


def load_model(checkpoint_path, num_classes, device='cpu'):
    """
    Load a saved model

    Args:
        checkpoint_path: Path to the checkpoint file
        num_classes: Number of output classes
        device: Device to load model on

    Returns:
        Loaded model and metadata
    """
    print(f"Loading model from: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    model_name = checkpoint['model_name']

    model = get_model(model_name, num_classes, pretrained=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    print(f"Model loaded successfully!")
    print(f"   Architecture: {model_name}")
    print(f"   Run: {checkpoint['run_name']}")
    print(f"   Saved: {checkpoint['timestamp'][:19]}")

    metadata = checkpoint.get('metadata', {})

    return model, metadata


def load_ensemble_from_checkpoints(convnext_path, efficientnet_path, deit_path,
                                   num_classes, device='cpu', weights=None, mode='average'):
    """
    Load an ensemble from three separate checkpoint files

    Args:
        convnext_path: Path to ConvNeXt checkpoint
        efficientnet_path: Path to EfficientNetV2 checkpoint
        deit_path: Path to DeiT checkpoint
        num_classes: Number of classes
        device: Device to load on
        weights: Ensemble weights
        mode: Ensemble mode

    Returns:
        EnsembleModel with loaded weights
    """
    print("Loading Ensemble from checkpoints...")
    print("="*60)

    model_convnext, _ = load_model(convnext_path, num_classes, device)
    model_efficientnet, _ = load_model(efficientnet_path, num_classes, device)
    model_deit, _ = load_model(deit_path, num_classes, device)

    ensemble = EnsembleModel(
        model_convnext,
        model_efficientnet,
        model_deit,
        weights=weights,
        mode=mode
    )

    ensemble.to(device)
    ensemble.eval()

    print("="*60)
    print("Ensemble loaded successfully!")

    return ensemble

def compare_models(num_classes=5):
    """
    Compare all three model architectures
    Returns comparison table
    """
    print("Comparing Model Architectures...")
    print("="*60)

    models_info = []

    for model_name in ['ConvNeXt-Base', 'EfficientNetV2-M', 'DeiT-Small']:
        model = get_model(model_name, num_classes, pretrained=False)
        info = get_model_info(model, model_name)
        models_info.append(info)

        print(f"\n{model_name}:")
        print(f"   Total Parameters: {info['total_parameters_M']}")
        print(f"   Trainable Parameters: {info['trainable_parameters_M']}")

    print("\n" + "="*60)

    return models_info

def freeze_backbone(model, freeze=True):
    """
    Freeze or unfreeze the backbone (all layers except the classifier head).
    Supports common timm architectures:
      - ConvNeXt / ViT / DeiT: model.head
      - EfficientNet / MobileNet: model.classifier
      - ResNet variants: model.fc
    """
    if not freeze:
        for p in model.parameters():
            p.requires_grad = True
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        print(f"Backbone unfrozen. Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M")
        return

    for p in model.parameters():
        p.requires_grad = False

    classifier_unfrozen = False

    if hasattr(model, 'head') and hasattr(model.head, 'parameters'):

        for p in model.head.parameters():
            p.requires_grad = True
        classifier_unfrozen = True

    if not classifier_unfrozen and hasattr(model, 'classifier') and hasattr(model.classifier, 'parameters'):
        for p in model.classifier.parameters():
            p.requires_grad = True
        classifier_unfrozen = True

    if not classifier_unfrozen and hasattr(model, 'fc') and hasattr(model.fc, 'parameters'):
        for p in model.fc.parameters():
            p.requires_grad = True
        classifier_unfrozen = True

    if not classifier_unfrozen:
        for name, p in model.named_parameters():
            if any(k in name.lower() for k in ['head', 'classifier', 'fc']):
                p.requires_grad = True
                classifier_unfrozen = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    if classifier_unfrozen:
        pct = (trainable / total * 100.0) if total > 0 else 0.0
        print(f"Backbone frozen. Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M ({pct:.2f}%)")
    else:
        print("WARNING: Could not identify classifier head. All layers remain frozen.")
        print(f"Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M")



Writing src/model_handler.py


## Step 5: Create Model Handler Module

This module (`src/model_handler.py`) provides:

### Key Functions:
- **`get_model()`**: Load individual models (ConvNeXt, EfficientNetV2, DeiT)
- **`EnsembleModel`**: Combine multiple models with weighted averaging
- **`save_model()`**: Save trained models with metadata
- **`load_model()`**: Load saved model checkpoints
- **`freeze_backbone()`**: Freeze/unfreeze layers for transfer learning

### Supported Architectures:
1. **ConvNeXt-Base** (88M parameters) - Excellent overall accuracy
2. **EfficientNetV2-M** (54M parameters) - Fast and efficient
3. **DeiT-Small** (22M parameters) - Vision transformer with attention

### Ensemble Modes:
- **Average**: Simple average of all predictions
- **Weighted**: Custom weights for each model
- **Max**: Take maximum probability across models

In [2]:
%%writefile src/predict.py
import json
from pathlib import Path
from typing import List, Optional, Dict, Tuple

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms

from src import model_handler


def load_classes(classes_path: str = "classes.json") -> List[str]:
    p = Path(classes_path)
    if not p.exists():

        return ["Bacterial leaf blight", "Brown spot", "Leaf smut"]
    return json.loads(p.read_text())


def _val_transform(img_size: int = 224):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

def _prep_batch(image_paths: List[str], img_size: int) -> torch.Tensor:
    tfm = _val_transform(img_size)
    imgs = []
    for p in image_paths:
        img = Image.open(p).convert("RGB")
        imgs.append(tfm(img))
    return torch.stack(imgs, dim=0)

def _topk_from_probs(probs: np.ndarray, class_names: List[str], k: int = 3):
    k = min(k, len(class_names))
    idx = np.argsort(-probs)[:k]
    return list(idx), [class_names[i] for i in idx], list(probs[idx])


def _scan_checkpoints(ckpt_dir: str = "models/checkpoints") -> List[Dict]:
    items = []
    for fp in Path(ckpt_dir).glob("*.pth"):
        try:
            ck = torch.load(fp, map_location="cpu")
            meta = ck.get("metadata", {})
            model_name = ck.get("model_name") or meta.get("model_name")
            best_acc = meta.get("best_val_accuracy")  # your Trainer stored this in percent
            items.append({
                "path": str(fp),
                "model_name": model_name,
                "best_val_accuracy": float(best_acc) if best_acc is not None else 0.0
            })
        except Exception:
            continue
    return items

def _best_overall_ckpt(ckpts: List[Dict]) -> Optional[Dict]:
    if not ckpts:
        return None
    return sorted(ckpts, key=lambda x: x.get("best_val_accuracy", 0.0), reverse=True)[0]

def _best_ckpt_per_arch(ckpts: List[Dict], arches: List[str]) -> Dict[str, Optional[Dict]]:
    result = {}
    for arch in arches:
        cands = [c for c in ckpts if (c.get("model_name") == arch)]
        if cands:
            result[arch] = sorted(cands, key=lambda x: x.get("best_val_accuracy", 0.0), reverse=True)[0]
        else:
            result[arch] = None
    return result


def _infer_model(model, image_paths: List[str], device: torch.device,
                 class_names: List[str], batch_size: int = 16, img_size: int = 224) -> pd.DataFrame:
    model.eval()
    records = []
    with torch.no_grad():
        for i in range(0, len(image_paths), batch_size):
            chunk = image_paths[i:i+batch_size]
            x = _prep_batch(chunk, img_size).to(device)
            logits = model(x)
            prob = F.softmax(logits, dim=1).cpu().numpy()
            pred_idx = prob.argmax(axis=1)
            for j, p in enumerate(chunk):
                idx = int(pred_idx[j])
                rec = {
                    "filepath": p,
                    "pred_idx": idx,
                    "pred_label": class_names[idx],
                    "confidence": float(prob[j, idx])
                }
                top_idx, top_lbl, top_val = _topk_from_probs(prob[j], class_names, k=3)
                for t, (lbl, val) in enumerate(zip(top_lbl, top_val), start=1):
                    rec[f"top{t}_label"] = lbl
                    rec[f"top{t}_prob"] = float(val)
                for ci, cname in enumerate(class_names):
                    rec[f"prob_{cname}"] = float(prob[j, ci])
                records.append(rec)
    return pd.DataFrame(records)


def predict_best_overall(image_paths: List[str], device: torch.device,
                         classes_path: str = "classes.json", img_size: int = 224,
                         ckpt_dir: str = "models/checkpoints") -> Tuple[pd.DataFrame, Dict]:
    class_names = load_classes(classes_path)
    num_classes = len(class_names)
    ckpts = _scan_checkpoints(ckpt_dir)
    best = _best_overall_ckpt(ckpts)
    if not best:
        raise RuntimeError("No checkpoints found in models/checkpoints")
    model, _ = model_handler.load_model(best["path"], num_classes=num_classes, device=device)
    df = _infer_model(model, image_paths, device, class_names, img_size=img_size)
    return df, {"model_used": best["model_name"], "best_val_accuracy": best["best_val_accuracy"]}

def predict_arch(arch_name: str, image_paths: List[str], device: torch.device,
                 classes_path: str = "classes.json", img_size: int = 224,
                 ckpt_dir: str = "models/checkpoints") -> Tuple[pd.DataFrame, Dict]:
    class_names = load_classes(classes_path)
    num_classes = len(class_names)
    ckpts = _scan_checkpoints(ckpt_dir)
    per_arch = _best_ckpt_per_arch(ckpts, [arch_name])
    entry = per_arch.get(arch_name)
    if not entry:
        raise RuntimeError(f"No checkpoint found for {arch_name} in {ckpt_dir}")
    model, _ = model_handler.load_model(entry["path"], num_classes=num_classes, device=device)
    df = _infer_model(model, image_paths, device, class_names, img_size=img_size)
    return df, {"model_used": arch_name, "best_val_accuracy": entry.get("best_val_accuracy")}

def predict_ensemble(image_paths: List[str], device: torch.device,
                     weights: Optional[List[float]] = None, mode: str = "weighted",
                     classes_path: str = "classes.json", img_size: int = 224,
                     ckpt_dir: str = "models/checkpoints",
                     logits_fusion: bool = False) -> Tuple[pd.DataFrame, Dict]:
    class_names = load_classes(classes_path)
    num_classes = len(class_names)
    ckpts = _scan_checkpoints(ckpt_dir)
    arches = ["ConvNeXt-Base", "EfficientNetV2-M", "DeiT-Small"]
    sel = _best_ckpt_per_arch(ckpts, arches)
    if not all(sel[a] for a in arches):
        missing = [a for a in arches if not sel[a]]
        raise RuntimeError(f"Missing checkpoints for: {missing}")

    if not logits_fusion:

        ensemble = model_handler.load_ensemble_from_checkpoints(
            convnext_path=sel["ConvNeXt-Base"]["path"],
            efficientnet_path=sel["EfficientNetV2-M"]["path"],
            deit_path=sel["DeiT-Small"]["path"],
            num_classes=num_classes,
            device=device,
            weights=weights or [1/3, 1/3, 1/3],
            mode=mode
        )
        df = _infer_model(ensemble, image_paths, device, class_names, img_size=img_size)
        meta = {"models_used": arches, "weights": weights or [1/3, 1/3, 1/3], "mode": mode, "fusion": "probabilities"}
        return df, meta


    models = []
    for a in arches:
        m, _ = model_handler.load_model(sel[a]["path"], num_classes=num_classes, device=device)
        m.eval()
        models.append(m)
    w = weights or [1/3, 1/3, 1/3]
    s = sum(w) or 1.0
    w = [wi / s for wi in w]

    records = []
    with torch.no_grad():
        for i in range(0, len(image_paths), 16):
            chunk = image_paths[i:i+16]
            x = _prep_batch(chunk, img_size).to(device)
            logits_sum = None
            for wi, m in zip(w, models):
                logits = m(x) * wi
                logits_sum = logits if logits_sum is None else logits_sum + logits
            prob = F.softmax(logits_sum, dim=1).cpu().numpy()
            pred_idx = prob.argmax(axis=1)
            for j, p in enumerate(chunk):
                idx = int(pred_idx[j])
                rec = {
                    "filepath": p,
                    "pred_idx": idx,
                    "pred_label": class_names[idx],
                    "confidence": float(prob[j, idx])
                }
                top_idx, top_lbl, top_val = _topk_from_probs(prob[j], class_names, k=3)
                for t, (lbl, val) in enumerate(zip(top_lbl, top_val), start=1):
                    rec[f"top{t}_label"] = lbl
                    rec[f"top{t}_prob"] = float(val)
                for ci, cname in enumerate(class_names):
                    rec[f"prob_{cname}"] = float(prob[j, ci])
                records.append(rec)
    df = pd.DataFrame(records)
    meta = {"models_used": arches, "weights": w, "mode": mode, "fusion": "logits"}
    return df, meta

Writing src/predict.py


## Step 6: Create Prediction Module

This module (`src/predict.py`) handles inference and predictions:

### Key Functions:
- **`predict_best_overall()`**: Use the single best-performing model
- **`predict_arch()`**: Use a specific architecture (ConvNeXt/EfficientNet/DeiT)
- **`predict_ensemble()`**: Combine predictions from multiple models

### Features:
- Automatic checkpoint scanning and selection
- Batch processing of multiple images
- Top-K predictions with probabilities
- Detailed probability distribution per class
- Support for both probability and logit fusion

### Output Format:
Returns a pandas DataFrame with:
- `filepath`: Image path
- `pred_label`: Predicted disease class
- `confidence`: Prediction confidence score
- `top1_label`, `top2_label`, `top3_label`: Top 3 predictions
- `prob_<disease>`: Probability for each disease class

In [1]:
%ls -lh models\checkpoints

 Volume in drive C is Acer
 Volume Serial Number is 2E61-F911

 Directory of c:\Users\csart\Downloads\Rice_Disease_Live


 Directory of c:\Users\csart\Downloads\Rice_Disease_Live\models\checkpoints

15-10-2025  14:07    <DIR>          .
15-10-2025  14:03    <DIR>          ..
15-10-2025  10:52       350,426,111 convnext_base_rice_v1_augmented_20251014_154540.pth
15-10-2025  10:51        86,733,029 deit_small_rice_v1_augmented_20251014_154738.pth
15-10-2025  10:51       213,064,345 efficientnetv2_m_rice_v1_augmented_20251014_154703.pth
               3 File(s)    650,223,485 bytes
               2 Dir(s)  42,619,326,464 bytes free


File Not Found


---

## Step 7: Verify Model Files

**IMPORTANT:** Before proceeding, ensure you have copied trained model files to `models/checkpoints/`.

### Expected Files:
```
models/checkpoints/
├── convnext_base_*.pth
├── efficientnetv2_m_*.pth
└── deit_small_*.pth
```

These files are generated from training in `model/backend/ML_Crop_Disease_Detection_Model.ipynb`.

Let's check if the files exist:

In [4]:
from src import predict
print("Classes:", predict.load_classes("classes.json"))
print("Scan:", predict._scan_checkpoints("models/checkpoints")[:3])

Classes: ['Bacterial leaf blight', 'Brown spot', 'Leaf smut']
Scan: [{'path': 'models\\checkpoints\\convnext_base_rice_v1_augmented_20251014_154540.pth', 'model_name': 'ConvNeXt-Base', 'best_val_accuracy': 95.83333333333334}, {'path': 'models\\checkpoints\\deit_small_rice_v1_augmented_20251014_154738.pth', 'model_name': 'DeiT-Small', 'best_val_accuracy': 95.83333333333334}, {'path': 'models\\checkpoints\\efficientnetv2_m_rice_v1_augmented_20251014_154703.pth', 'model_name': 'EfficientNetV2-M', 'best_val_accuracy': 70.83333333333334}]


## Step 8: Test Prediction Module

Testing if the prediction module can:
- Load disease classes from `classes.json`
- Scan and detect trained model checkpoints

This validates that our setup is correct before launching the full application.

In [6]:
from pathlib import Path
import torch

for fp in Path("models/checkpoints").glob("*.pth"):
    ck = torch.load(fp, map_location="cpu")
    model_name = ck.get("model_name") or ck.get("metadata", {}).get("model_name")
    print(fp.name, "->", model_name)

convnext_base_rice_v1_augmented_20251014_154540.pth -> ConvNeXt-Base
deit_small_rice_v1_augmented_20251014_154738.pth -> DeiT-Small
efficientnetv2_m_rice_v1_augmented_20251014_154703.pth -> EfficientNetV2-M


## Step 9: Verify Model Architecture Names

Checking that each checkpoint file has the correct model architecture name. This is crucial for:
- Automatic model loading
- Correct architecture selection
- Ensemble creation

Each `.pth` file should contain metadata with the model name (ConvNeXt-Base, EfficientNetV2-M, or DeiT-Small).

In [7]:
import torch

path = "models/checkpoints/efficientnetv2_m_rice_v1_augmented_20251014_154703.pth"
ck = torch.load(path, map_location="cpu")
print(ck.keys())

if "metadata" in ck:
    print(ck["metadata"])
    print("Model name:", ck["model_name"])
else:
    print("No metadata found")

dict_keys(['model_state_dict', 'model_name', 'run_name', 'timestamp', 'metadata'])
{'dataset_version': 'rice_v1_augmented', 'hyperparameters': {'phase1_epochs': 5, 'phase2_epochs': 15, 'phase1_lr': 0.001, 'phase2_lr': 5e-05, 'optimizer': 'AdamW', 'criterion': 'CrossEntropyLoss', 'phase1_best_acc': 70.83333333333334, 'two_phase_training': True}, 'best_val_accuracy': 70.83333333333334, 'best_val_f1': 0.7022875816993465, 'best_epoch': 2, 'total_epochs': 10, 'training_time_minutes': 0.43049491246541344, 'final_train_acc': 100.0, 'final_val_acc': 62.5}
Model name: EfficientNetV2-M


## Step 10: Inspect Model Metadata (EfficientNetV2-M)

Examining the contents and metadata of the EfficientNetV2-M checkpoint file to verify:
- Model state dictionary
- Training metadata (accuracy, date, etc.)
- Architecture information
- Number of classes

In [8]:
import torch

path = "models/checkpoints/deit_small_rice_v1_augmented_20251014_154738.pth"
ck = torch.load(path, map_location="cpu")
print(ck.keys())

if "metadata" in ck:
    print(ck["metadata"])
    print("Model name:", ck["model_name"])
else:
    print("No metadata found")

dict_keys(['model_state_dict', 'model_name', 'run_name', 'timestamp', 'metadata'])
{'dataset_version': 'rice_v1_augmented', 'hyperparameters': {'phase1_epochs': 5, 'phase2_epochs': 15, 'phase1_lr': 0.001, 'phase2_lr': 5e-05, 'optimizer': 'AdamW', 'criterion': 'CrossEntropyLoss', 'phase1_best_acc': 79.16666666666666, 'two_phase_training': True}, 'best_val_accuracy': 95.83333333333334, 'best_val_f1': 0.9581699346405229, 'best_epoch': 1, 'total_epochs': 12, 'training_time_minutes': 0.4096132477124532, 'final_train_acc': 100.0, 'final_val_acc': 91.66666666666666}
Model name: DeiT-Small


## Step 11: Inspect Model Metadata (DeiT-Small)

Examining the DeiT (Vision Transformer) model checkpoint to verify its structure and metadata.

In [9]:
%%writefile app.py
import os
from pathlib import Path
from datetime import datetime
import json
import requests

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src import model_handler, predict

DEBUG = True
Path("models/checkpoints").mkdir(parents=True, exist_ok=True)
Path("logs/predictions").mkdir(parents=True, exist_ok=True)
Path("logs/gemini").mkdir(parents=True, exist_ok=True)

def _get_device():
    import torch
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _parse_weights(s: str):
    if not s:
        return None
    try:
        parts = [float(x) for x in s.split(",")]
        ssum = sum(parts) or 1.0
        parts = [x / ssum for x in parts]
        if len(parts) != 3:
            return None
        return parts
    except Exception as e:
        if DEBUG: print(f"[DEBUG] parse_weights error: {e}")
        return None

def _first_image_prob_plot(df: pd.DataFrame):
    if df is None or df.empty:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.text(0.5, 0.5, "No predictions", ha="center", va="center")
        ax.axis("off")
        return fig

    row = df.iloc[0]
    cols = [c for c in df.columns if c.startswith("prob_")]
    if not cols:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.text(0.5, 0.5, "No probability columns", ha="center", va="center")
        ax.axis("off")
        return fig

    labels = [c.replace("prob_", "") for c in cols]
    vals = row[cols].astype(float).values
    order = vals.argsort()[::-1]
    labels = [labels[i] for i in order]
    vals = vals[order]

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.barh(labels[::-1], vals[::-1], color="#4C78A8")
    ax.set_xlim(0, 1.0)
    ax.set_xlabel("Probability")
    ax.set_title(f"Top probabilities for: {Path(row['filepath']).name}")
    for i, v in enumerate(vals[::-1]):
        ax.text(v + 0.01, i, f"{v:.3f}", va="center")
    fig.tight_layout()
    return fig

def top_n_prob_pie(df: pd.DataFrame, n=3):
    if df is None or df.empty:
        return None
    row = df.iloc[0]
    prob_cols = [c for c in df.columns if c.startswith("prob_")]
    labels = [c.replace("prob_", "") for c in prob_cols]
    values = row[prob_cols].astype(float).values
    top_idx = values.argsort()[::-1][:n]
    top_labels = [labels[i] for i in top_idx]
    top_values = [values[i] for i in top_idx]

    fig, ax = plt.subplots(figsize=(6, 7))
    ax.pie(top_values, labels=top_labels, autopct="%1.1f%%", startangle=140, colors=plt.cm.Set3.colors)
    ax.set_title("Top Predictions Distribution")
    return fig

def prediction_confidence_gauge(df: pd.DataFrame):
    if df is None or df.empty:
        return None
    
    row = df.iloc[0]
    prob_cols = [c for c in df.columns if c.startswith("prob_")]
    values = row[prob_cols].astype(float).values
    max_val = values.max() 
    
    theta = np.linspace(-np.pi/2, np.pi/2, 100)
    fig, ax = plt.subplots(figsize=(5, 3), subplot_kw={'projection': 'polar'})
    ax.set_theta_zero_location('S') 
    ax.set_theta_direction(-1) 

    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.set_yticks([])
    ax.set_xticks([])
    
    thresholds = [0.33, 0.66, 1.0]
    colors = ['red', 'yellow', 'green']
    start = -np.pi/2
    for i, t in enumerate(thresholds):
        end = -np.pi/2 + t * np.pi
        ax.barh(1, width=end-start, left=start, height=1.5, color=colors[i], alpha=0.3, edgecolor='k')
        start = end
    
    needle_angle = -np.pi/2 + max_val * np.pi
    ax.arrow(needle_angle, 0, 0, 1.5, width=0.03, head_width=0.1, head_length=0.2, fc='black', ec='black')
    
    ax.text(-np.pi/3, 1.8, "Low", ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(0, 1.9, "Medium", ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(np.pi/3, 1.8, "High", ha='center', va='center', fontsize=10, fontweight='bold')
    
    ax.set_title(f"Prediction Confidence: {max_val:.2f}", va='bottom', fontsize=12, fontweight='bold')
    
    return fig


def get_gemini_advice(disease_name: str):
    api_key = os.getenv("GEMINI_API_KEY", "")
    if not api_key or disease_name.lower() == "healthy":
        return '{"causes": [], "treatment": [], "prevention": []}', None 

    prompt = (
        f"Rice leaf disease: {disease_name}.\n"
        "Provide treatment advice, causes, and preventive measures in JSON format:\n"
        '{"causes": [...], "treatment": [...], "prevention": [...]}'
    )

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "gemini-2.5-flash",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    }

    try:
        response = requests.post(
            "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions",
            headers=headers,
            json=payload,
            timeout=30
        )
        response.raise_for_status()
        data = response.json()
        text_output = data.get("choices", [{}])[0].get("message", {}).get("content", "{}")

        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        log_file_txt = Path(f"logs/gemini/gemini_{disease_name}_{ts}.txt")
        with open(log_file_txt, "w", encoding="utf-8") as f:
            f.write(f"Prompt:\n{prompt}\n\nResponse:\n{text_output}")
        if DEBUG: print(f"[DEBUG] Gemini response logged to {log_file_txt}")

        # Save CSV automatically if valid JSON
        try:
            advice_json = json.loads(text_output)
            df_csv = pd.DataFrame({
                "Causes": advice_json.get("causes", []),
                "Treatment": advice_json.get("treatment", []),
                "Prevention": advice_json.get("prevention", [])
            })
            csv_path = Path(f"logs/gemini/gemini_{disease_name}_{ts}.csv")
            df_csv.to_csv(csv_path, index=False)
            if DEBUG: print(f"[DEBUG] Gemini advice CSV saved to {csv_path}")
        except Exception:
            pass

        return text_output, None
    except Exception as e:
        return f'Error calling Gemini API: {str(e)}', None

def run_inference(files, model_choice, weights_text, mode):
    if not files:
        return None, None, pd.DataFrame(), None, "No files uploaded.", "", "Healthy"

    device = _get_device()
    paths = [f.name if hasattr(f, "name") else f for f in files]
    weights = _parse_weights(weights_text) if model_choice.startswith("Ensemble") else None

    try:
        if model_choice == "Best Overall":
            df, meta = predict.predict_best_overall(paths, device=device, classes_path="classes.json")
        elif model_choice in ["ConvNeXt-Base", "EfficientNetV2-M", "DeiT-Small"]:
            df, meta = predict.predict_arch(model_choice, paths, device=device, classes_path="classes.json")
        elif model_choice == "Ensemble":
            df, meta = predict.predict_ensemble(
                paths, device=device, weights=weights, mode=mode,
                classes_path="classes.json", logits_fusion=False
            )
        else:
            df, meta = predict.predict_ensemble(
                paths, device=device, weights=weights, mode=mode,
                classes_path="classes.json", logits_fusion=True
            )
    except Exception as e:
        return None, None, pd.DataFrame(), None, f"Error: {str(e)}", "", "Healthy"

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_csv = Path("logs/predictions") / f"preds_{model_choice.replace(' ', '_').lower()}_{ts}.csv"
    df.to_csv(out_csv, index=False)
    if DEBUG: print(f"[DEBUG] Prediction CSV saved to {out_csv}")

    gallery = paths
    prob_plot = _first_image_prob_plot(df)
    pie_chart = top_n_prob_pie(df)
    confidence_gauge = prediction_confidence_gauge(df)
    summary = f"Model selection: {model_choice}\nMeta: {meta}\nSaved CSV: {out_csv}"

    top_prob_cols = [c for c in df.columns if c.startswith("prob_")]
    top_idx = df[top_prob_cols].iloc[0].astype(float).values.argmax()
    top_label = top_prob_cols[top_idx].replace("prob_", "")
    disease_status = "Healthy" if top_label.lower() == "healthy" else top_label

    return gallery, prob_plot, pie_chart, confidence_gauge, df, str(out_csv), summary, disease_status, top_label

def get_treatment_advice(disease_name):
    text, _ = get_gemini_advice(disease_name)
    return text

# --- Gradio UI ---
with gr.Blocks(title="VeAg: Vacant Vectors Model", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        <div style="text-align:center">
            <h1>VeAg by Vacant Vectors: Rice Leaf Disease Detection & Treatment Advice Model</h1>
            <p style="font-size:16px;">Upload Rice leaf images and choose a model to get predictions.</p>
            <p style="font-size:16px;">Disclaimer: This tool provides automated rice leaf disease predictions and treatment suggestions for educational and informational purposes only. It is not a substitute for professional agricultural advice for now.</p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Inputs")
            files_u = gr.Files(label="Upload images", file_count="multiple", type="filepath")
            model_choice = gr.Dropdown(
                label="Model",
                choices=[
                    "Best Overall",
                    "ConvNeXt-Base",
                    "EfficientNetV2-M",
                    "DeiT-Small",
                    "Ensemble",
                    "Ensemble (logits)"
                ],
                value="Best Overall"
            )
            weights_text = gr.Textbox(
                label="Ensemble Weights (comma-separated, optional)",
                placeholder="0.45,0.10,0.45"
            )
            mode = gr.Dropdown(
                label="Ensemble Mode",
                choices=["average", "weighted"],
                value="weighted"
            )
            btn_predict = gr.Button("Predict", variant="primary")
            btn_gemini = gr.Button("Get Treatment Advice", variant="secondary")

        with gr.Column(scale=2):
            gr.Markdown("### Preview and Results")
            gallery = gr.Gallery(label="Uploaded Images", show_label=True)
            prob_plot = gr.Plot(label="Probability Bar Chart (first image)")
            pie_chart = gr.Plot(label="Top-N Predictions Pie Chart")
            confidence_gauge = gr.Plot(label="Prediction Confidence Gauge")
            out_df = gr.Dataframe(label="Predictions")
            out_file = gr.File(label="Download Predictions CSV")
            out_txt = gr.Textbox(label="Summary", lines=6)
            disease_status = gr.Textbox(label="Disease Status", lines=2)
            gemini_output = gr.Textbox(label="Treatment Advice (JSON)", lines=20)

    btn_predict.click(
        run_inference,
        inputs=[files_u, model_choice, weights_text, mode],
        outputs=[gallery, prob_plot, pie_chart, confidence_gauge, out_df, out_file, out_txt, disease_status, disease_status]
    )

    btn_gemini.click(
        get_treatment_advice,
        inputs=[disease_status],
        outputs=[gemini_output]
    )

if __name__ == "__main__":
    port = int(os.getenv("PORT", "7860"))
    demo.launch(server_name="0.0.0.0", server_port=port, pwa=True)


Writing app.py


---

## Step 12: Create Gradio Web Application

This creates `app.py` - the main web interface with:

### Features:
1. **Image Upload**: Single or multiple crop leaf images
2. **Model Selection**:
   - Best Overall (automatic)
   - Individual models (ConvNeXt, EfficientNetV2, DeiT)
   - Ensemble with custom weights
   - Ensemble with logit fusion
3. **Visualizations**:
   - Probability bar charts
   - Top-N predictions pie chart
   - Confidence gauge meter
4. **AI Treatment Advisor**: Get treatment advice via Google Gemini API
5. **Export**: Download predictions as CSV

### Ensemble Modes:
- **Average**: Equal weight to all models
- **Weighted**: Custom weights (e.g., 0.45, 0.10, 0.45)

### Treatment Advice:
Integrates with Google Gemini API to provide:
- Causes of the disease
- Treatment recommendations
- Prevention measures

**Note:** Treatment advice requires `GEMINI_API_KEY` in `.env` file.

In [11]:
#Testing
import os
import requests
import json
from dotenv import load_dotenv

# --- Set your Gemini API key ---
api_key = os.getenv("GEMINI_API_KEY", "YOUR_KEY_HERE")
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Please set your GEMINI_API_KEY in the .env file.")

# --- Sample prompt for testing ---
prompt = (
    "Rice leaf disease: Brown Spot.\n"
    "Provide treatment advice, causes, and preventive measures in JSON format:\n"
    '{"causes": [...], "treatment": [...], "prevention": [...]}'
)

# --- Headers ---
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# --- Payload ---
payload = {
    "model": "gemini-2.5-flash",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
}

# --- Call Gemini API ---
response = requests.post(
    "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions",
    headers=headers,
    json=payload
)

# --- Handle response ---
if response.status_code == 200:
    data = response.json()
    text_output = data.get("choices", [{}])[0].get("message", {}).get("content", "")
    print("Raw output from Gemini API:\n", text_output)

    # Optional: parse JSON
    try:
        advice_json = json.loads(text_output.strip())
        print("\nParsed JSON:")
        for key, items in advice_json.items():
            print(f"{key.capitalize()}:")
            for i, item in enumerate(items, 1):
                print(f"  {i}. {item}")
    except Exception as e:
        print("Could not parse JSON:", e)
else:
    print(f"Error {response.status_code}: {response.text}")


Raw output from Gemini API:
 ```json
{
  "causes": [
    "Fungal pathogen: Bipolaris oryzae (also known as Helminthosporium oryzae).",
    "High humidity and prolonged leaf wetness.",
    "Warm temperatures (20-30°C) during crop growth.",
    "Nutrient deficiencies, particularly potassium (K), silicon (Si), magnesium (Mg), calcium (Ca), and zinc (Zn).",
    "Poor soil fertility and imbalanced nutrient application (e.g., excessive nitrogen without balanced K and P).",
    "Acidic or alkaline soil conditions.",
    "Drought stress or other environmental stresses.",
    "Susceptible rice varieties.",
    "Presence of infected plant debris from previous crops (inoculum source)."
  ],
  "treatment": [
    "Apply appropriate fungicides as soon as symptoms appear, such as Propiconazole, Tebuconazole, Mancozeb, Carbendazim, or Azoxystrobin. Consult local agricultural extension for specific recommendations and dosage.",
    "Improve nutrient management: Apply balanced fertilizers to correct def

---

## Step 13: Test Gemini API Integration (Optional)

**Optional Testing**: This cell tests the Google Gemini API integration for treatment advice.

### Setup Required:
1. Create a `.env` file in the current directory
2. Add your Gemini API key: `GEMINI_API_KEY=your_key_here`
3. Get API key from: https://ai.google.dev/

### What This Tests:
- API connectivity
- JSON response parsing
- Treatment advice format

**Note:** If you don't have a Gemini API key, skip this cell. The main prediction functionality will still work without it.

In [12]:
import os
from pathlib import Path

# Creating logs/predictions directory
Path("logs/predictions").mkdir(parents=True, exist_ok=True)

print("Directory 'logs/predictions' created successfully!\n")



Directory 'logs/predictions' created successfully!



---

## Step 14: Create Logging Directories

Setting up directories for storing:
- **`logs/predictions/`**: CSV files of all predictions made
- **`logs/gemini/`**: Treatment advice responses (JSON and CSV format)

These directories are automatically used by the application to save results.

In [13]:
import os
from pathlib import Path

# Creating logs/predictions directory
Path("logs/gemini").mkdir(parents=True, exist_ok=True)

print("Directory 'logs/gemini' created successfully!\n")

Directory 'logs/gemini' created successfully!



## Step 15: Create Gemini Logs Directory

Creating directory for Gemini API treatment advice logs.

In [1]:
try:
    import app
    app.demo.close()
except Exception:
    pass

import importlib, app
importlib.reload(app)
app.demo.launch(share=True, inline=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://166101f1b66c58bb72.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## Step 16: Launch Gradio Application

**Final Step!** Launch the Gradio web interface.

### What Happens:
1. Loads all trained models from `models/checkpoints/`
2. Starts web server on port 7860
3. Creates a shareable link (optional)
4. Opens interface in browser or displays inline

### Using the Interface:
1. **Upload Images**: Click "Upload images" and select crop leaf photos
2. **Choose Model**: Select from dropdown (Best Overall recommended)
3. **Adjust Weights** (optional): For ensemble, set custom weights
4. **Click "Predict"**: Get disease predictions with visualizations
5. **Get Treatment** (optional): Click "Get Treatment Advice" for AI recommendations
6. **Download Results**: Export predictions as CSV

### Interface URL:
- Local: `http://localhost:7860`
- Shareable: Generated link (if `share=True`)

### Stopping:
- In notebook: Interrupt kernel
- In terminal: Press `Ctrl+C`

---

## All Setup Complete!

You now have a fully functional crop disease detection system with:
- Multi-model ensemble predictions
- Interactive web interface
- AI-powered treatment recommendations
- Comprehensive visualizations
- Export capabilities

### Next Steps:
1. Test with sample images
2. Adjust ensemble weights for best results
3. (Optional) Integrate with VeAg platform backend
4. Adapt for other crops by training new models

---

## 📝 Notes

### Current Implementation:
- **Crop**: Rice leaf diseases
- **Classes**: Bacterial leaf blight, Brown spot, Leaf smut
- **Models**: ConvNeXt-Base, EfficientNetV2-M, DeiT-Small

### Adapting to Other Crops:
1. Train new models in `model/backend/` with your crop dataset
2. Copy `.pth` files to `models/checkpoints/`
3. Update `classes.json` with your disease categories
4. Rerun this notebook

### Troubleshooting:
- **No models found**: Copy `.pth` files to `models/checkpoints/`
- **Class mismatch**: Update `classes.json` to match trained models
- **CUDA errors**: System will auto-fallback to CPU
- **Gemini API errors**: Check API key in `.env` file

See `README.md` for detailed documentation.